In [ ]:
print('Importing libraries...')
import json
import os
from pathlib import Path
import warnings 

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim

from scripts.env import set_deterministic_behaviour
from scripts.refactoring_caching import cache_validation
from scripts.dataset import CachedDataset
from scripts.dino_encoder import build_dino_encoder
from scripts.metrics import update_model_output_dict, calculate_metrics

warnings.filterwarnings("ignore")

PWD = Path.cwd()
print(f"PWD: {PWD}")

In [ ]:
# General
SEED = 0
EXPERIMENT_NAME = f"DinoV3_encoder_pretraining_{SEED}"

# Data Paths
ANNOTATIONS_PATH = PWD / 'config/reformatted_annotations_frames.json'
DATASET_PATH = PWD.parent / 'dataset/endoscapes/'
CACHED_IMAGES_PATH = PWD / 'cached_images_single'

# Parameters to create the dataset
TEMPORAL = False
FORCE_RECACHE = False

# Batch size
BATCH_SIZE = 4 # 16

# Encoder specific
DINO_PRETRAINED_WEIGHTS_PATH = './weights/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'
LORA = None
PARTIAL_TRAINING_BLOCKS = 6
kwargs = {  'lora': LORA,
            'partial_training_blocks': PARTIAL_TRAINING_BLOCKS,
            'imnt_weights_path': DINO_PRETRAINED_WEIGHTS_PATH}

# Training specific
EPOCHS = 1
BACKBONE_LR = 1e-4
HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-2
CLASS_WEIGHTS = [3.19852941, 4.46153846, 2.79518072]

In [ ]:
# Check CUDA availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")

# For reproducible results
set_deterministic_behaviour(SEED)

In [ ]:
# These are declared in scripts/refactoring_caching.py
# IMAGE_SIZE = (384, 384)
# DATASET_MEAN = (0.454315, 0.290313, 0.299898)
# DATASET_STD = (0.167318, 0.156652, 0.150197)

cache_validation(   DATASET_PATH,
                    CACHED_IMAGES_PATH,
                    ANNOTATIONS_PATH,
                    temporal = TEMPORAL,
                    force_recache = FORCE_RECACHE)

In [ ]:
# Paths
train_set_path = CACHED_IMAGES_PATH / 'train'
val_set_path = CACHED_IMAGES_PATH / 'val'
test_set_path = CACHED_IMAGES_PATH / 'test'

# Datasets
dataset_train = CachedDataset(  train_set_path,
                                label_criterion = (None, 'hard'))
dataset_val =   CachedDataset(  val_set_path,
                                label_criterion = (None, 'hard'))
dataset_test =  CachedDataset(  test_set_path,
                                label_criterion = (None, 'hard'))

# Dataloaders
train_dataloader =  DataLoader( dataset_train,
                                batch_size = BATCH_SIZE,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = True)

val_dataloader =    DataLoader( dataset_val,
                                batch_size = BATCH_SIZE,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = False)

test_dataloader =   DataLoader( dataset_test,
                                batch_size = BATCH_SIZE,
                                pin_memory = True,
                                drop_last= False,
                                shuffle = False)

In [ ]:
# Init DinoV3 Backbone
model = build_dino_encoder(**kwargs)

# Separate parameter groups for adjusted learning rate
backbone_params = []
head_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if name.startswith("head"):
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer = optim.AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)
model.to(device)

class_weights = torch.tensor(CLASS_WEIGHTS).to(device) # weights, specific to BCE, taken from official endoscapes implementation repository
bce_loss = nn.BCEWithLogitsLoss(weight=class_weights).to(device)

In [ ]:
# MODEL TRAINING AND EVALUATION
results_dict = {}

best_bacc_across_epochs = -1.0
best_epoch = 0

for epoch in range(EPOCHS):
        print(f"Epoch: {epoch+1:02}/{EPOCHS:02}")
        print("Training")
        train_loss_sum = 0.0
        len_train_loader = len(train_dataloader)
        train_output_dict = {'C1':  {'probs':     [],
                                     'preds':     []},
                             'C2':  {'probs':     [],
                                     'preds':     []},
                             'C3':  {'probs':     [],
                                     'preds':     []},
                             'labels':            [],
                             'vid_ids':           [],
                             'frame_ids':         []}
        model.train()
        for idx, (images, labels, vid_id, frame_id) in enumerate(train_dataloader):
                print(f'\r{idx+1}/{len_train_loader}', end='', flush=True)

                images, labels = images.to(device), labels.to(device)
                torch.cuda.synchronize()

                optimizer.zero_grad()

                output = model(images)

                train_loss_per_batch = bce_loss(output, labels)

                train_loss_per_batch.backward()
                optimizer.step()

                # Populate the output dict with probs and preds per batch per class
                train_output_dict = update_model_output_dict(output, train_output_dict)
                train_output_dict['labels'].append(labels.detach().cpu())
                train_output_dict['vid_ids'].append(vid_id)
                train_output_dict['frame_ids'].append(frame_id)
                train_loss_sum += train_loss_per_batch.item()

        results, train_output_dict = calculate_metrics(train_output_dict)

        avg_train_loss = train_loss_sum / len_train_loader
        results['loss'] = round(avg_train_loss, 4)

        print(f"\n--- Training Metrics ---")
        print(f"Train Avg Accuracy              {results['avg_accuracy']:.4f}")
        print(f"Train Avg BAcc                  {results['avg_bacc']:.4f}")
        print(f"Train mAP                       {results['mAP']:.4f}")
        print(f"Train Loss:                     {results['loss']:.4f}\n")
        print(f"Train C1 Accuracy               {results['accuracy_C1']:.4f}")
        print(f"Train C2 Accuracy               {results['accuracy_C2']:.4f}")
        print(f"Train C3 Accuracy               {results['accuracy_C3']:.4f}\n")
        print(f"Train C1 Balanced Accuracy:     {results['bal_accuracy_C1']:.4f}")
        print(f"Train C2 Balanced Accuracy:     {results['bal_accuracy_C2']:.4f}")
        print(f"Train C3 Balanced Accuracy:     {results['bal_accuracy_C3']:.4f}\n")
        print(f"Train C1 AP:                    {results['ap_C1']:.4f}")
        print(f"Train C2 AP:                    {results['ap_C2']:.4f}")
        print(f"Train C3 AP:                    {results['ap_C3']:.4f}")
        print(f"------------------------\n")
        results_dict[f"Epoch {epoch+1} Train"] = results

        print('Validation')
        val_loss_sum = 0.0
        len_val_loader = len(val_dataloader)
        val_output_dict = {     'C1':  {'probs':     [],
                                        'preds':     []},
                                'C2':  {'probs':     [],
                                        'preds':     []},
                                'C3':  {'probs':     [],
                                        'preds':     []},
                                'labels':            [],
                                'vid_ids':           [],
                                'frame_ids':         []}
        model.eval()
        with torch.inference_mode():
                for idx, (images, labels, vid_id, frame_id) in enumerate(val_dataloader):
                        print(f'\r{idx+1}/{len_val_loader}', end='', flush=True)
                        images, labels = images.to(device), labels.to(device)
                        torch.cuda.synchronize()

                        output = model(images)

                        val_loss_per_batch = bce_loss(output, labels)

                        val_output_dict = update_model_output_dict(output, val_output_dict)
                        val_output_dict['labels'].append(labels.detach().cpu())
                        val_output_dict['vid_ids'].append(vid_id)
                        val_output_dict['frame_ids'].append(frame_id)
                        val_loss_sum += val_loss_per_batch.item()

        results, val_output_dict = calculate_metrics(val_output_dict)
        avg_val_loss = val_loss_sum / len_val_loader
        results['loss'] = round(avg_val_loss, 4)
        print(f"\n--- Validation Metrics ---")
        print(f"Val Avg Accuracy              {results['avg_accuracy']:.4f}")
        print(f"Val Avg BAcc                  {results['avg_bacc']:.4f}")
        print(f"Val mAP                       {results['mAP']:.4f}")
        print(f"Val Loss:                     {results['loss']:.4f}\n")
        print(f"Val C1 Accuracy               {results['accuracy_C1']:.4f}")
        print(f"Val C2 Accuracy               {results['accuracy_C2']:.4f}")
        print(f"Val C3 Accuracy               {results['accuracy_C3']:.4f}\n")
        print(f"Val C1 Balanced Accuracy:     {results['bal_accuracy_C1']:.4f}")
        print(f"Val C2 Balanced Accuracy:     {results['bal_accuracy_C2']:.4f}")
        print(f"Val C3 Balanced Accuracy:     {results['bal_accuracy_C3']:.4f}\n")
        print(f"Val C1 AP:                    {results['ap_C1']:.4f}")
        print(f"Val C2 AP:                    {results['ap_C2']:.4f}")
        print(f"Val C3 AP:                    {results['ap_C3']:.4f}")
        print(f"------------------------\n")

        results['saved'] = {    'C1': { 'probs':     val_output_dict['C1']['probs'].tolist(),
                                        'preds':     val_output_dict['C1']['preds'].tolist()},
                                'C2': { 'probs':     val_output_dict['C2']['probs'].tolist(),
                                        'preds':     val_output_dict['C2']['preds'].tolist()},
                                'C3': { 'probs':     val_output_dict['C3']['probs'].tolist(),
                                        'preds':     val_output_dict['C3']['preds'].tolist()},
                                'labels':            val_output_dict['labels'].tolist(),
                                'vid_ids':           val_output_dict['vid_ids'].tolist(),
                                'frame_ids':         val_output_dict['frame_ids'].tolist()}
        results_dict[f"Epoch {epoch+1} Val"] = results

        # Save results
        with open(PWD / 'results' / f'{EXPERIMENT_NAME}_results.json', 'w') as file:
                json.dump(results_dict, file, indent=4)

        # Save weights of the best epoch
        if results['avg_bacc'] >= best_bacc_across_epochs:
                best_bacc_across_epochs = results['avg_bacc']
                best_epoch = epoch+1
                print(f"New best result (Epoch {best_epoch}), saving weights...")
                weights_path = Path.cwd() / 'weights'
                checkpoint_dir = os.path.join(weights_path, f'{EXPERIMENT_NAME}.pt')
                torch.save(model.state_dict(), checkpoint_dir)
        else:
                print('\n')

print(f"Testing @ epoch {best_epoch}")
test_loss_sum = 0.0
len_test_loader = len(test_dataloader)
test_output_dict = {    'C1':  {'probs':     [],
                                'preds':     []},
                        'C2':  {'probs':     [],
                                'preds':     []},
                        'C3':  {'probs':     [],
                                'preds':     []},
                        'labels':            [],
                        'vid_ids':           [],
                        'frame_ids':         []}
checkpoint = torch.load(checkpoint_dir, map_location=device)
model.load_state_dict(checkpoint)
model.to(device)
model.eval()
with torch.inference_mode():
    for idx, (images, labels, vid_id, frame_id) in enumerate(test_dataloader):
        print(f'\r{idx+1}/{len_test_loader}', end='', flush=True)
        images, labels = images.to(device), labels.to(device)
        torch.cuda.synchronize()
        
        output = model(images)

        test_loss_per_batch = bce_loss(output, labels)

        test_output_dict = update_model_output_dict(output, test_output_dict)
        test_output_dict['labels'].append(labels.detach().cpu())
        test_output_dict['vid_ids'].append(vid_id)
        test_output_dict['frame_ids'].append(frame_id)
        test_loss_sum += test_loss_per_batch.item()

results, test_output_dict = calculate_metrics(test_output_dict)
avg_test_loss = test_loss_sum / len_test_loader
results['loss'] = round(avg_test_loss, 4)

print(f"\n--- Testing Metrics ---")
print(f"Test Avg Accuracy              {results['avg_accuracy']:.4f}")
print(f"Test Avg BAcc                  {results['avg_bacc']:.4f}")
print(f"Test mAP                       {results['mAP']:.4f}")
print(f"Test Loss:                     {results['loss']:.4f}\n")
print(f"Test C1 Accuracy               {results['accuracy_C1']:.4f}")
print(f"Test C2 Accuracy               {results['accuracy_C2']:.4f}")
print(f"Test C3 Accuracy               {results['accuracy_C3']:.4f}\n")
print(f"Test C1 Balanced Accuracy:     {results['bal_accuracy_C1']:.4f}")
print(f"Test C2 Balanced Accuracy:     {results['bal_accuracy_C2']:.4f}")
print(f"Test C3 Balanced Accuracy:     {results['bal_accuracy_C3']:.4f}\n")
print(f"Test C1 AP:                    {results['ap_C1']:.4f}")
print(f"Test C2 AP:                    {results['ap_C2']:.4f}")
print(f"Test C3 AP:                    {results['ap_C3']:.4f}")
print(f"------------------------\n")
results['saved'] = {'C1': { 'probs':     test_output_dict['C1']['probs'].tolist(),
                            'preds':     test_output_dict['C1']['preds'].tolist()},
                    'C2': { 'probs':     test_output_dict['C2']['probs'].tolist(),
                            'preds':     test_output_dict['C2']['preds'].tolist()},
                    'C3': { 'probs':     test_output_dict['C3']['probs'].tolist(),
                            'preds':     test_output_dict['C3']['preds'].tolist()},
                    'labels':            test_output_dict['labels'].tolist(),
                    'vid_ids':           test_output_dict['vid_ids'].tolist(),
                    'frame_ids':         test_output_dict['frame_ids'].tolist()}

results_dict[f"Epoch {best_epoch} Test"] = results
with open(PWD / 'results' / f'{EXPERIMENT_NAME}_results.json', 'w') as file:
    json.dump(results_dict, file, indent=4)
